In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\ASUS\Desktop\Resume_classifier\resume_dataset\Resume\Resume.csv")

print(df['Category'].value_counts())

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


In [2]:
# Keep useful categories only
keep = [
    "FINANCE",
    "BANKING",
    "ACCOUNTANT",
    "BUSINESS-DEVELOPMENT",
    "INFORMATION-TECHNOLOGY",
    "ENGINEERING",
    "DESIGNER",
    "DIGITAL-MEDIA",
    "ARTS"
]

df = df[df["Category"].isin(keep)].copy()

print(df["Category"].value_counts())
print("Total rows:", len(df))

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ACCOUNTANT                118
FINANCE                   118
ENGINEERING               118
BANKING                   115
DESIGNER                  107
ARTS                      103
DIGITAL-MEDIA              96
Name: count, dtype: int64
Total rows: 1015


In [3]:
import re

def classify_role(text, category):
    text = str(text).lower()

    # Direct mapping
    if category in ["FINANCE", "BANKING", "ACCOUNTANT", "BUSINESS-DEVELOPMENT"]:
        return "Data Analyst"

    if category in ["DESIGNER", "DIGITAL-MEDIA", "ARTS"]:
        return "UI/UX Designer"

    ml_keywords = [
        "python","pandas","numpy","scikit","sklearn","tensorflow",
        "keras","pytorch","machine learning","deep learning",
        "nlp","data science","opencv","jupyter"
    ]

    web_keywords = [
        "html","css","javascript","react","node","angular",
        "bootstrap","php","frontend","backend","web development"
    ]

    if category in ["INFORMATION-TECHNOLOGY", "ENGINEERING"]:

        ml_score = sum(word in text for word in ml_keywords)
        web_score = sum(word in text for word in web_keywords)

        if ml_score >= 2 and ml_score > web_score:
            return "ML Engineer"
        else:
            return "Web Developer"

    return None

In [4]:
df["Final_Label"] = df.apply(
    lambda x: classify_role(x["Resume_str"], x["Category"]),
    axis=1
)

print(df["Final_Label"].value_counts())

Final_Label
Data Analyst      471
UI/UX Designer    306
Web Developer     235
ML Engineer         3
Name: count, dtype: int64


In [5]:
ml_samples = [
    "Python TensorFlow PyTorch Deep Learning NLP Machine Learning Pandas NumPy SQL",
    "Machine Learning Engineer with Python Scikit-learn TensorFlow Keras NLP",
    "Built predictive models using Python Pandas NumPy Scikit-learn Matplotlib",
    "Deep Learning Computer Vision OpenCV CNN TensorFlow PyTorch Python",
    "Natural Language Processing NLP Transformers BERT Python PyTorch",
    "Data Science Machine Learning Regression Classification Python SQL",
    "MLOps Docker AWS FastAPI TensorFlow Machine Learning Deployment",
]

In [6]:
import random

synthetic_ml = []

for _ in range(150):
    text = " ".join(random.sample(ml_samples, k=3))
    synthetic_ml.append({
        "Resume_str": text,
        "Final_Label": "ML Engineer"
    })

ml_df = pd.DataFrame(synthetic_ml)

In [7]:
real_df = df[["Resume_str", "Final_Label"]]

final_df = pd.concat([real_df, ml_df], ignore_index=True)

print(final_df["Final_Label"].value_counts())

Final_Label
Data Analyst      471
UI/UX Designer    306
Web Developer     235
ML Engineer       153
Name: count, dtype: int64


In [8]:
import os

os.makedirs("data", exist_ok=True)

final_df.to_csv("data/final_dataset.csv", index=False)

In [9]:
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("data/final_dataset.csv")

le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["Final_Label"])

print(df[["Final_Label", "label_encoded"]].drop_duplicates())

        Final_Label  label_encoded
0    UI/UX Designer              2
107   Web Developer              3
227    Data Analyst              0
578     ML Engineer              1


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["Resume_str"],
    df["label_encoded"],
    test_size=0.2,
    random_state=42,
    stratify=df["label_encoded"]
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

Train size: 932
Test size : 233


In [11]:
import numpy as np
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

In [12]:
train_df = pd.DataFrame({
    "text": X_train.values,
    "label": y_train.values
})

test_df = pd.DataFrame({
    "text": X_test.values,
    "label": y_test.values
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [13]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [14]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/932 [00:00<?, ? examples/s]

Map:   0%|          | 0/233 [00:00<?, ? examples/s]

In [15]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_dir="./logs",

    load_best_model_at_end=True,

    fp16=False
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [19]:
trainer.train()

C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.269874,0.922747,0.922545
2,No log,0.280328,0.931330,0.931151


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=466, training_loss=0.3287868909058141, metrics={'train_runtime': 651.7738, 'train_samples_per_second': 2.86, 'train_steps_per_second': 0.715, 'total_flos': 61732009500672.0, 'train_loss': 0.3287868909058141, 'epoch': 2.0})

In [20]:
trainer.evaluate()

C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: on_train_begin must be called before on_evaluate

In [21]:
trainer.save_model("resume_classifier_model")
tokenizer.save_pretrained("resume_classifier_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('resume_classifier_model\\tokenizer_config.json',
 'resume_classifier_model\\tokenizer.json')